In [1]:
# ============================================================================
# CELL 1: Import Libraries
# ============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


In [2]:
# ============================================================================
# CELL 2: Load Cleaned Datasets
# ============================================================================
print("\n" + "="*80)
print("LOADING CLEANED DATASETS")
print("="*80)

# Pastikan olist_cleaned.csv sudah dibuat ulang dari Notebook 1
# dan sudah berisi 'customer_unique_id'
df_fraud = pd.read_csv('outputs/data/fraud_cleaned.csv')
df_olist = pd.read_csv('outputs/data/olist_cleaned.csv')

print(f"Fraud dataset shape: {df_fraud.shape}")
print(f"Olist dataset shape: {df_olist.shape}")

# Convert timestamp back to datetime for olist
df_olist['order_purchase_timestamp'] = pd.to_datetime(df_olist['order_purchase_timestamp'])


LOADING CLEANED DATASETS
Fraud dataset shape: (284807, 32)
Olist dataset shape: (115035, 18)


In [3]:
# ============================================================================
# CELL 3: Definisi Fungsi, Fitur, & Timestep (V-SINTA1)
# ============================================================================
print("\n" + "="*80)
print("DEFINISI FUNGSI DAN FITUR (V-SINTA1)")
print("="*80)

TIMESTEP = 2 # Kita gunakan 2 sebagai minimum absolut
STRIDE = 1

# --- DAFTAR FITUR KAGGLE ---
# (Pastikan fitur ini ada di df_fraud setelah diproses di SEL 4)
fraud_feature_cols_raw = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time_sin', 'Time_cos']

# --- DAFTAR FITUR OLIST (V-SINTA1, NO LEAKAGE) ---
# Ini adalah fitur "sekunder" yang AMAN (tidak ada contekan)
final_olist_feature_cols = [
    'payment_installments',
    'freight_value',
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos'
]
olist_metadata_cols = ['customer_unique_id', 'Class', 'anomaly_type']

# --- FUNGSI WINDOWING (KAGGLE) ---
def create_sequences_kaggle(data, labels, timestep, stride):
    sequences, sequence_labels = [], []
    for i in range(0, len(data) - timestep + 1, stride):
        sequences.append(data[i:i + timestep])
        sequence_labels.append(labels[i + timestep - 1])
    return np.array(sequences), np.array(sequence_labels)

# --- FUNGSI WINDOWING (OLIST PER USER) ---
def create_sequences_per_user(df_group, feature_cols, timestep):
    features = df_group[feature_cols].values
    labels = df_group['Class'].values
    anomaly_types = df_group['anomaly_type'].values
    seqs, seq_labels, seq_types = [], [], []
    if len(features) < timestep: return None
    for i in range(len(features) - timestep + 1):
        seqs.append(features[i:i + timestep])
        seq_labels.append(labels[i + timestep - 1])
        seq_types.append(anomaly_types[i + timestep - 1])
    return seqs, seq_labels, seq_types

def process_dataframe_to_sequences(df, feature_cols, timestep):
    all_seqs, all_labels, all_types = [], [], []
    # Pastikan df tidak kosong
    if df.empty:
        print("Warning: process_dataframe_to_sequences menerima dataframe kosong.")
        # Kembalikan array kosong dengan bentuk yang benar (meskipun 0 sampel)
        n_features = len(feature_cols)
        empty_seqs = np.empty((0, timestep, n_features), dtype=np.float32)
        empty_labels = np.empty((0,), dtype=np.int64)
        empty_types = np.empty((0,), dtype=object)
        return empty_seqs, empty_labels, empty_types
        
    grouped = df.groupby('customer_unique_id')
    for _, group in grouped:
        group = group.sort_values(by='order_purchase_timestamp')
        result = create_sequences_per_user(group, feature_cols, timestep)
        if result is not None:
            all_seqs.extend(result[0])
            all_labels.extend(result[1])
            all_types.extend(result[2])
            
    if not all_seqs: # Jika list masih kosong setelah loop
        n_features = len(feature_cols)
        empty_seqs = np.empty((0, timestep, n_features), dtype=np.float32)
        empty_labels = np.empty((0,), dtype=np.int64)
        empty_types = np.empty((0,), dtype=object)
        return empty_seqs, empty_labels, empty_types

    return np.array(all_seqs), np.array(all_labels), np.array(all_types)

print("✓ Definisi fungsi, fitur (V-SINTA1), dan TIMESTEP=2 telah diatur.")


DEFINISI FUNGSI DAN FITUR (V-SINTA1)
✓ Definisi fungsi, fitur (V-SINTA1), dan TIMESTEP=2 telah diatur.


In [4]:
# ============================================================================
# CELL 4: Proses Data Kaggle
# ============================================================================
print("\n" + "="*80)
print("KAGGLE DATASET - PREPROCESSING")
print("="*80)

# 1. Buat Fitur
df_fraud_processed = df_fraud.copy()
df_fraud_processed['Time_hours'] = df_fraud_processed['Time'] / 3600
df_fraud_processed['Time_sin'] = np.sin(2 * np.pi * df_fraud_processed['Time_hours'] / 24)
df_fraud_processed['Time_cos'] = np.cos(2 * np.pi * df_fraud_processed['Time_hours'] / 24)
# Scale 'Amount'
df_fraud_processed['Amount'] = StandardScaler().fit_transform(df_fraud_processed[['Amount']])

fraud_feature_cols = [col for col in fraud_feature_cols_raw if col in df_fraud_processed.columns]
print(f"Fitur Kaggle yang digunakan: {fraud_feature_cols}")

# 2. Scaling
print("Melakukan scaling pada data Kaggle...")
X_fraud = df_fraud_processed[fraud_feature_cols].values
y_fraud = df_fraud_processed['Class'].values
scaler_fraud = StandardScaler()
X_fraud_scaled = scaler_fraud.fit_transform(X_fraud)
joblib.dump(scaler_fraud, 'outputs/models/scaler_fraud.pkl')

# 3. Windowing
print("Membuat sekuens untuk data Kaggle...")
X_fraud_seq, y_fraud_seq = create_sequences_kaggle(X_fraud_scaled, y_fraud, TIMESTEP, STRIDE)
print(f"Kaggle sequence shape: {X_fraud_seq.shape}")

# 4. Splitting (Chronological)
print("Melakukan Chronological Split pada data Kaggle...")
test_size = int(len(X_fraud_seq) * 0.15); val_size = int(len(X_fraud_seq) * 0.15)
train_size = len(X_fraud_seq) - val_size - test_size
X_train_fraud, y_train_fraud = X_fraud_seq[:train_size], y_fraud_seq[:train_size]
X_val_fraud, y_val_fraud = X_fraud_seq[train_size : train_size + val_size], y_fraud_seq[train_size : train_size + val_size]
X_test_fraud, y_test_fraud = X_fraud_seq[train_size + val_size :], y_fraud_seq[train_size + val_size :]

# Pisahkan data latih (hanya normal)
train_normal_mask_fraud = (y_train_fraud == 0)
X_train_fraud_normal = X_train_fraud[train_normal_mask_fraud]
y_train_fraud_normal = y_train_fraud[train_normal_mask_fraud]

# 5. Saving (Kaggle)
print("Menyimpan data Kaggle...")
np.save('outputs/data/X_train_fraud.npy', X_train_fraud_normal)
np.save('outputs/data/y_train_fraud.npy', y_train_fraud_normal)
np.save('outputs/data/X_val_fraud.npy', X_val_fraud)
np.save('outputs/data/y_val_fraud.npy', y_val_fraud)
np.save('outputs/data/X_test_fraud.npy', X_test_fraud)
np.save('outputs/data/y_test_fraud.npy', y_test_fraud)
print("✓ Data Kaggle disimpan.")


KAGGLE DATASET - PREPROCESSING
Fitur Kaggle yang digunakan: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Time_sin', 'Time_cos']
Melakukan scaling pada data Kaggle...
Membuat sekuens untuk data Kaggle...
Kaggle sequence shape: (284806, 2, 31)
Melakukan Chronological Split pada data Kaggle...
Menyimpan data Kaggle...
✓ Data Kaggle disimpan.


In [5]:
# ============================================================================
# CELL 5: Olist - Persiapan Data Normal & Split
# ============================================================================
print("\n" + "="*80)
print("OLIST DATASET - PERSIAPAN DATA NORMAL")
print("="*80)

# 1. Buat Fitur Dasar (Siklikal)
df_olist_processed = df_olist.copy()
df_olist_processed['hour_sin'] = np.sin(2 * np.pi * df_olist_processed['hour'] / 24)
df_olist_processed['hour_cos'] = np.cos(2 * np.pi * df_olist_processed['hour'] / 24)
df_olist_processed['dow_sin'] = np.sin(2 * np.pi * df_olist_processed['dayofweek'] / 7)
df_olist_processed['dow_cos'] = np.cos(2 * np.pi * df_olist_processed['dayofweek'] / 7)

# 2. Filter Pengguna (HANYA pengguna dengan >= TIMESTEP transaksi)
print(f"Memfilter pengguna dengan >= {TIMESTEP} transaksi...")
user_trx_counts = df_olist_processed['customer_unique_id'].value_counts()
users_with_enough_data = user_trx_counts[user_trx_counts >= TIMESTEP].index
df_normal_base = df_olist_processed[
    df_olist_processed['customer_unique_id'].isin(users_with_enough_data)
].copy()
print(f"Pengguna yang valid untuk sekuens: {len(users_with_enough_data)}")
print(f"Dataframe dasar normal (sebelum split): {df_normal_base.shape}")

# 3. User-Based Split (PADA PENGGUNA NORMAL)
print("Melakukan User-Based Split pada pengguna normal...")
unique_users = df_normal_base['customer_unique_id'].unique()
np.random.shuffle(unique_users)
train_users, test_val_users = train_test_split(unique_users, test_size=0.3, random_state=42)
val_users, test_users = train_test_split(test_val_users, test_size=0.5, random_state=42)

# Buat dataframe normal
df_train_normal_base = df_normal_base[df_normal_base['customer_unique_id'].isin(train_users)]
df_val_normal_base = df_normal_base[df_normal_base['customer_unique_id'].isin(val_users)]
df_test_normal_base = df_normal_base[df_normal_base['customer_unique_id'].isin(test_users)]

print(f"Train users: {len(train_users)}, Val users: {len(val_users)}, Test users: {len(test_users)}")
print("✓ Data normal Olist telah dipisahkan.")


OLIST DATASET - PERSIAPAN DATA NORMAL
Memfilter pengguna dengan >= 2 transaksi...
Pengguna yang valid untuk sekuens: 14020
Dataframe dasar normal (sebelum split): (35698, 22)
Melakukan User-Based Split pada pengguna normal...
Train users: 9814, Val users: 2103, Test users: 2103
✓ Data normal Olist telah dipisahkan.


In [6]:
# ============================================================================
# GANTI CELL 6: Olist - Fungsi Injeksi Anomali (V-SINTA1-FINAL)
# ============================================================================
print("\n" + "="*80)
print("OLIST DATASET - FUNGSI INJEKSI ANOMALI (V-SINTA1-FINAL)")
print("="*80)

def inject_anomalies_sinta1_final(df_normal, target_fraud_ratio=0.10):
    """
    Menyuntikkan anomali TEMPORAL (halus) dan STATIS yang sebenarnya.
    """
    df_injected = df_normal.copy()
    df_injected['Class'] = 0
    df_injected['anomaly_type'] = 'normal'
    
    n_anomalies_to_add = int(len(df_injected) * target_fraud_ratio)
    n_static = int(n_anomalies_to_add / 2)
    n_temporal = n_anomalies_to_add - n_static
    
    # --- 1. Injeksi Anomali Statis (Sama seperti sebelumnya) ---
    # Anomali: 'freight_value' yang sangat tinggi (5x lipat)
    static_candidates_idx = df_injected.index
    if n_static > 0 and len(static_candidates_idx) >= n_static:
        print(f"Menyuntikkan {n_static} anomali STATIS (freight * 5)...")
        static_inject_idx = np.random.choice(static_candidates_idx, n_static, replace=False)
        
        df_injected.loc[static_inject_idx, 'freight_value'] = df_injected.loc[static_inject_idx, 'freight_value'] * 5
        df_injected.loc[static_inject_idx, 'Class'] = 1
        df_injected.loc[static_inject_idx, 'anomaly_type'] = 'static'

    # --- 2. Injeksi Anomali Sekuensial (LEBIH HALUS) ---
    # Anomali: Perubahan drastis 'payment_installments' dari 1 -> 3
    
    # Urutkan berdasarkan pengguna dan waktu
    df_injected = df_injected.sort_values(by=['customer_unique_id', 'order_purchase_timestamp'])
    
    # Temukan 'payment_installments' dari transaksi sebelumnya (t-1)
    df_injected['prev_installments'] = df_injected.groupby('customer_unique_id')['payment_installments'].shift(1)
    
    # Cari kandidat:
    # 1. Transaksi saat ini (t) adalah 'normal'
    # 2. Transaksi sebelumnya (t-1) ada (bukan NaN)
    # 3. Transaksi sebelumnya (t-1) punya 1 installment
    # 4. Transaksi saat ini (t) punya 1 installment (untuk diubah)
    temporal_candidates_idx = df_injected[
        (df_injected['Class'] == 0) &
        (df_injected['prev_installments'].notna()) &
        (df_injected['prev_installments'] == 1) &
        (df_injected['payment_installments'] == 1) # Pastikan kita mengubah urutan [1, 1]
    ].index
    
    n_temporal_actual = min(n_temporal, len(temporal_candidates_idx))
    
    if n_temporal_actual > 0:
        print(f"Menyuntikkan {n_temporal_actual} anomali SEKUENSIAL HALUS (Installment 1 -> 3)...")
        temporal_inject_idx = np.random.choice(
            temporal_candidates_idx, n_temporal_actual, replace=False
        )
        
        # Lakukan anomali: Ubah 'payment_installments' di transaksi 't' menjadi 3
        df_injected.loc[temporal_inject_idx, 'payment_installments'] = 3
        
        # Labeli transaksi ini sebagai anomali 'temporal'
        df_injected.loc[temporal_inject_idx, 'Class'] = 1
        df_injected.loc[temporal_inject_idx, 'anomaly_type'] = 'temporal'

    # --- 3. Cleanup (Anti-Target Leakage) ---
    # Hapus fitur helper 'prev_installments'
    df_injected = df_injected.drop(columns=['prev_installments'], errors='ignore')
    
    print("✓ Injeksi anomali V-SINTA1-FINAL selesai.")
    return df_injected

print("✓ Fungsi 'inject_anomalies_sinta1_final' didefinisikan.")


OLIST DATASET - FUNGSI INJEKSI ANOMALI (V-SINTA1-FINAL)
✓ Fungsi 'inject_anomalies_sinta1_final' didefinisikan.


In [7]:
# ============================================================================
# GANTI CELL 7: Olist - Terapkan Injeksi (FINAL), Scaling, & Windowing
# ============================================================================
print("\n" + "="*80)
print("OLIST DATASET - INJEKSI (SINTA1-FINAL), SCALING, WINDOWING")
print("="*80)

# 1. Terapkan Injeksi Anomali V-SINTA1-FINAL ke Val dan Test
print("Menyuntikkan anomali SINTA1-FINAL ke df_val dan df_test...")
df_val_mixed = inject_anomalies_sinta1_final(df_val_normal_base, target_fraud_ratio=0.15)
df_test_mixed = inject_anomalies_sinta1_final(df_test_normal_base, target_fraud_ratio=0.15)

print(f"Distribusi Val baru: \n{df_val_mixed['Class'].value_counts(normalize=True)}")
print(f"Distribusi Test baru: \n{df_test_mixed['Class'].value_counts(normalize=True)}")

# Tambahkan kolom 'Class' dan 'anomaly_type' ke data latih
print("\nMenambahkan label 'normal' ke data latih...")
df_train_normal_base['Class'] = 0
df_train_normal_base['anomaly_type'] = 'normal'

# 2. Scaling (Fit HANYA pada data train normal)
print("\nMelakukan Scaling (Fit HANYA di data Train Normal)...")
scaler_olist = StandardScaler()
scaler_olist.fit(df_train_normal_base[final_olist_feature_cols])
joblib.dump(scaler_olist, 'outputs/models/scaler_olist.pkl')

# Terapkan scaler ke semua data split
df_train_normal_base[final_olist_feature_cols] = scaler_olist.transform(df_train_normal_base[final_olist_feature_cols])
df_val_mixed[final_olist_feature_cols] = scaler_olist.transform(df_val_mixed[final_olist_feature_cols])
df_test_mixed[final_olist_feature_cols] = scaler_olist.transform(df_test_mixed[final_olist_feature_cols])
print("✓ Olist Scaling selesai.")

# 3. Terapkan Windowing
print("\nMembuat sekuens Olist...")
X_train_olist_normal, y_train_olist_normal, _ = process_dataframe_to_sequences(
    df_train_normal_base, final_olist_feature_cols, TIMESTEP
)
X_val_olist, y_val_olist, val_types_olist = process_dataframe_to_sequences(
    df_val_mixed, final_olist_feature_cols, TIMESTEP
)
X_test_olist, y_test_olist, types_test_olist = process_dataframe_to_sequences(
    df_test_mixed, final_olist_feature_cols, TIMESTEP
)

# 4. Simpan Data
print("Menyimpan data Olist (V-SINTA1-FINAL)...")
np.save('outputs/data/X_train_olist.npy', X_train_olist_normal)
np.save('outputs/data/y_train_olist.npy', y_train_olist_normal)
np.save('outputs/data/X_val_olist.npy', X_val_olist)
np.save('outputs/data/y_val_olist.npy', y_val_olist)
np.save('outputs/data/X_test_olist.npy', X_test_olist)
np.save('outputs/data/y_test_olist.npy', y_test_olist)
olist_test_info = {'anomaly_types': types_test_olist}
np.save('outputs/data/olist_test_info.npy', olist_test_info, allow_pickle=True)
print("✓ Semua data Olist (V-SINTA1-FINAL) telah diproses dan disimpan!")


OLIST DATASET - INJEKSI (SINTA1-FINAL), SCALING, WINDOWING
Menyuntikkan anomali SINTA1-FINAL ke df_val dan df_test...
Menyuntikkan 402 anomali STATIS (freight * 5)...
Menyuntikkan 403 anomali SEKUENSIAL HALUS (Installment 1 -> 3)...
✓ Injeksi anomali V-SINTA1-FINAL selesai.
Menyuntikkan 398 anomali STATIS (freight * 5)...
Menyuntikkan 398 anomali SEKUENSIAL HALUS (Installment 1 -> 3)...
✓ Injeksi anomali V-SINTA1-FINAL selesai.
Distribusi Val baru: 
Class
0    0.850121
1    0.149879
Name: proportion, dtype: float64
Distribusi Test baru: 
Class
0    0.850038
1    0.149962
Name: proportion, dtype: float64

Menambahkan label 'normal' ke data latih...

Melakukan Scaling (Fit HANYA di data Train Normal)...
✓ Olist Scaling selesai.

Membuat sekuens Olist...
Menyimpan data Olist (V-SINTA1-FINAL)...
✓ Semua data Olist (V-SINTA1-FINAL) telah diproses dan disimpan!


In [8]:
# ============================================================================
# CELL 8: Summary & Simpan Konfigurasi
# ============================================================================
print("\n" + "="*80)
print("PREPROCESSING SUMMARY (V-SINTA1)")
print("="*80)

summary_data = {
    'Dataset': ['Fraud - Train', 'Fraud - Val', 'Fraud - Test',
                'Olist - Train', 'Olist - Val', 'Olist - Test'],
    'Shape': [X_train_fraud_normal.shape, X_val_fraud.shape, X_test_fraud.shape,
              X_train_olist_normal.shape, X_val_olist.shape, X_test_olist.shape],
    'Normal': [len(y_train_fraud_normal), (y_val_fraud==0).sum(), (y_test_fraud==0).sum(),
               len(y_train_olist_normal), (y_val_olist==0).sum(), (y_test_olist==0).sum()],
    'Fraud': [0, (y_val_fraud==1).sum(), (y_test_fraud==1).sum(),
              0, (y_val_olist==1).sum(), (y_test_olist==1).sum()]
}
df_summary = pd.DataFrame(summary_data)
print("\n", df_summary.to_string(index=False))

print("\nMenyimpan konfigurasi...")
config = {
    'timestep': TIMESTEP,
    'stride': STRIDE,
    'n_features_fraud': len(fraud_feature_cols),
    'n_features_olist': len(final_olist_feature_cols),
    'fraud_feature_cols': fraud_feature_cols,
    'olist_feature_cols': final_olist_feature_cols
}
np.save('outputs/data/config.npy', config, allow_pickle=True)
print("✓ Configuration saved!")
print("\n" + "="*80)
print("PREPROCESSING V-SINTA1 COMPLETED!")
print("="*80)


PREPROCESSING SUMMARY (V-SINTA1)

       Dataset           Shape  Normal  Fraud
Fraud - Train (198982, 2, 31)  198982      0
  Fraud - Val  (42720, 2, 31)   42664     56
 Fraud - Test  (42720, 2, 31)   42668     52
Olist - Train   (15205, 2, 6)   15205      0
  Olist - Val    (3268, 2, 6)    2617    651
 Olist - Test    (3205, 2, 6)    2581    624

Menyimpan konfigurasi...
✓ Configuration saved!

PREPROCESSING V-SINTA1 COMPLETED!


In [9]:
# SEL VERIFIKASI
print(f"Bentuk X_train_olist_normal yang disimpan: {X_train_olist_normal.shape}")
print(f"Bentuk y_train_olist_normal yang disimpan: {y_train_olist_normal.shape}")

Bentuk X_train_olist_normal yang disimpan: (15205, 2, 6)
Bentuk y_train_olist_normal yang disimpan: (15205,)
